# PARALLEL job: test-side embeddings for blocking v3
The train side is already done and saved (`MLC'26/work_emb`: train embeddings + kNN; measured recall 0.9503 -> **0.9784**).
This notebook does the same for **test**: encode 11.7M test records with `multilingual-e5-small`, then within-country GPU kNN.
- Output on Drive `MLC'26/work_emb/`: `test_emb.f16` (+ `test_emb_ids.npy`) and `knn_test_0..7.parquet`.
- About the time on a T4: encode **~2.2 h** (the tokenizer on 2 CPUs is the bottleneck), kNN ~20 min. **~2.5 h total.**
- **Resumable:** the encode runs in 8 chunks (~25 min each); every finished chunk is copied to Drive at once. After a crash / dead session /
  another account: run cells 1-4 again and it continues from the last finished chunk. At most one chunk is lost.
- **Run cells 1-4 one by one. Not "Run all".** Then re-run cell 5 for status. Keep the tab open and the laptop awake.

**Account note:** this account needs *Editor* access to the shared `MLC'26` folder, plus a shortcut named `MLC'26` in My Drive.

In [ ]:
# ---- Cell 1: config + mount + install (one Drive-auth click) ----
D = "/content/drive/MyDrive/MLC'26"      # shared folder, added to *this* account's My Drive as a shortcut named MLC'26
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, shutil
_free = shutil.disk_usage("/content/drive").free / 1e9
# Files count against the account that CREATES them (even inside the shared MLC'26), so THIS account's free space matters.
print(f"this account's free Drive space: {_free:.1f} GB  (V3 main needs ~6 GB, V3 helper ~7 GB; empty the Drive trash if it is lower)")
assert os.path.isdir(D), f"{D} not found: in Drive, open 'Shared with me' -> MLC'26 -> 'Add shortcut to My Drive'"
assert os.path.isdir(f"{D}/work_v2") and os.path.isdir(f"{D}/code_v2") and os.path.isdir(f"{D}/models"), "MLC'26 is missing work_v2 / code_v2 / models"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    print(r.stdout[-6000:], r.stderr[-3000:])
sh(f'pip install -q --no-index -f "{D}/wheels" rapidfuzz anyascii 2>/dev/null || pip install -q rapidfuzz anyascii')
sh("python -c 'import rapidfuzz, anyascii, xgboost, torch; print(\"ok\", xgboost.__version__, torch.cuda.is_available())'")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv; free -g | head -2; df -h /content | tail -1")

## Cell 2: restore what this job needs (test records, code) - about 2 min

In [ ]:
sh(f"""
mkdir -p /content/work /content/ber/src "{D}/work_emb"
cp -n "{D}"/work/test_records.parquet /content/work/
cp -n "{D}"/work_emb/test_emb.f16 "{D}"/work_emb/test_emb_ids.npy /content/work/ 2>/dev/null || true
cp -n "{D}"/work_emb/test_emb_c*.f16 /content/work/ 2>/dev/null || true   # finished chunks from an earlier session
cp -n "{D}"/work_emb/knn_test_*.parquet /content/work/ 2>/dev/null || true
ls -la /content/work; df -h /content | tail -1
""")

## Cell 3: write the code (verified locally end to end on a real-data sample)

In [ ]:
%%writefile /content/ber/src/common.py
"""Paths + shared helpers. Every stage reads/writes parquet under WORK so re-runs skip finished stages."""
import os
import time
from pathlib import Path

import polars as pl

HERE = Path(__file__).resolve().parent
DATA = Path(os.environ.get("BER_DATA", HERE.parents[2] / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"))
WORK = Path(os.environ.get("BER_WORK", HERE.parents[2] / "work"))
OUT = Path(os.environ.get("BER_OUT", HERE.parents[2] / "output"))
WORK.mkdir(parents=True, exist_ok=True)
SPLITS = ("train", "test")
# Version tag for everything a model run writes (p*, xgb_*, comp*, ce*, val_*). "" = v2 names (unchanged);
# "_v3" = blocking v3. FEAT = which feature files the models read (feat2 = v2, feat3 = v3 with kNN pairs).
TAG = os.environ.get("BER_TAG", "")
FEAT = os.environ.get("BER_FEAT", "feat2")


def encode_ids(col):
    """'S2-681193310' -> 2_681193310 (int64). Source digit * 1e9 + number."""
    c = pl.col(col)
    return (c.str.slice(1, 1).cast(pl.Int64) * 1_000_000_000 + c.str.slice(3).cast(pl.Int64))


def decode_ids(col):
    c = pl.col(col)
    return pl.concat_str([pl.lit("S"), (c // 1_000_000_000).cast(pl.Utf8), pl.lit("-"),
                          (c % 1_000_000_000).cast(pl.Utf8)])


def done(path):
    return Path(path).exists()


class timer:
    def __init__(self, msg):
        self.msg = msg

    def __enter__(self):
        self.t = time.time()
        print(f"> {self.msg}", flush=True)

    def __exit__(self, *a):
        print(f"  {self.msg}: {time.time() - self.t:.0f}s", flush=True)


def f05_scores(s1_all, pred, truth):
    """s1_all: [s1] every S1 scored (singletons count). pred/truth: [s1, id] pairs.
    Per S1: F0.5 = 1.25*tp / (0.25*n_true + n_pred); both empty -> 1."""
    tp =pred.join(truth, on=["s1", "id"], how="inner").group_by("s1").len("tp")
    npred = pred.group_by("s1").len("np")
    ntrue = truth.group_by("s1").len("nt")
    df = (s1_all.select("s1").join(tp, on="s1", how="left").join(npred, on="s1", how="left")
          .join(ntrue, on="s1", how="left").fill_null(0))
    f = pl.when((pl.col("np") == 0) & (pl.col("nt") == 0)).then(1.0).otherwise(
        1.25 * pl.col("tp") / (0.25 * pl.col("nt") + pl.col("np")).clip(lower_bound=1e-9))
    return df.with_columns(f.alias("f"))


def f05(s1_all, pred, truth):
    return f05_scores(s1_all, pred, truth)["f"].mean()

In [ ]:
%%writefile /content/ber/src/s2_block.py
"""Stage 2: blocking. Keys (all within country): see keys(). Blocks bigger than caps are dropped (generic keys).
Out per S1 bucket: [s1, id, nk0..nk8 = #shared keys per type, sc, rk, ncand_raw], top-K per S1."""
import sys

import polars as pl

from common import SPLITS, WORK, done, timer

CAP1, CAP23 = 20, 100   # max S1 / S2+S3 records sharing a key
PARTS = 32              # key-hash partitions -> bounded RAM
NKT = 9
KW = [3, 1, 1, 1, 1, 3, 3, 1, 1]  # rank weight per key type


RARE = {"atoks": 3, "ntoks": 3, "stoks": 2}
KCHUNKS = 16  # key emission in id chunks -> bounded RAM


def ntoks(lf):
    return lf.with_columns(pl.col("core").str.split(" ").list.eval(
        pl.element().filter(pl.element().str.len_chars() >= 3)).alias("ntoks"))


def token_df(lf, col):
    """Document frequency per (country, token) over the whole split; tokens seen once are useless keys."""
    return (lf.select("id", "country", pl.col(col).list.unique().alias("t")).explode("t").drop_nulls("t")
            .group_by("country", "t").len("df").filter(pl.col("df") >= 2).collect())


def rare(lf, col, n, df):
    """Per record keep the n rarest tokens of list column `col`."""
    t = lf.select("id", "country", pl.col(col).list.unique().alias("t")).explode("t").drop_nulls("t")
    return (t.join(df.lazy(), on=["country", "t"]).sort("id", "df", "t").group_by("id")
            .agg(pl.col("t").head(n).alias(col)))


def keys(lf, dfs):
    """kt: 0 core name | 1 number x rare addr token | 2 rare name tok x rare addr tok | 3 rare addr token pair
           4 rare name token pair | 5 name skeleton | 6 name without spaces (web/handle names)
           7 rare skeleton tok x rare addr tok | 8 rare skeleton tok x number"""
    lf = ntoks(lf)
    base = lf.select("id", "src", "country", "core", "skel", "nosp", pl.col("nums").list.head(3))
    for col, n in RARE.items():
        base = base.join(rare(lf, col, n, dfs[col]), on="id", how="left")

    def emit(kt, cols, frame):
        return frame.select("id", "src", pl.concat_str(["country", pl.lit(str(kt)), *cols], separator="|")
                            .hash().alias("k"), pl.lit(kt, pl.Int8).alias("kt"))

    def pairs(col):
        a = base.select("id", "src", "country", pl.col(col).alias("a")).explode("a")
        b = base.select("id", pl.col(col).alias("b")).explode("b")
        return a.join(b, on="id").filter(pl.col("a") < pl.col("b"))

    def cross(c1, c2):
        return base.explode(c1).explode(c2).drop_nulls([c1, c2])

    return pl.concat([
        emit(0, ["core"], base.filter(pl.col("core") != "")),
        emit(1, ["nums", "atoks"], cross("nums", "atoks")),
        emit(2, ["ntoks", "atoks"], cross("ntoks", "atoks")),
        emit(3, ["a", "b"], pairs("atoks")),
        emit(4, ["a", "b"], pairs("ntoks")),
        emit(5, ["skel"], base.filter(pl.col("skel").str.len_chars() >= 4)),
        emit(6, ["nosp"], base.filter(pl.col("nosp").str.len_chars() >= 6)),
        emit(7, ["stoks", "atoks"], cross("stoks", "atoks")),
        emit(8, ["stoks", "nums"], cross("stoks", "nums")),
    ]).unique(["id", "k"])


BUCKETS = 8   # S1-id buckets: every later stage works one bucket at a time
TOPK = int(__import__("os").environ.get("BER_TOPK", 60))   # candidates kept per S1


def cand_path(split, j):
    return WORK / f"{split}_cand2_{j}.parquet"


def block(split):
    if all(done(cand_path(split, j)) for j in range(BUCKETS)):
        return
    kfs = [WORK / f"{split}_keys2_{c}.parquet" for c in range(KCHUNKS)]
    if not all(done(f) for f in kfs):
        lf = ntoks(pl.scan_parquet(WORK / f"{split}_norm2.parquet").with_columns(pl.col("country").cast(pl.Utf8)))
        with timer(f"token df {split}"):
            dfs = {col: token_df(lf, col) for col in RARE}
        for c, f in enumerate(kfs):
            if not done(f):
                with timer(f"keys {split} chunk {c}"):
                    keys(lf.filter(pl.col("id") % KCHUNKS == c), dfs).collect().write_parquet(f)
        del dfs
    kf = kfs
    parts = []
    for p in range(PARTS):
        pf = WORK / f"{split}_cand2_p{p}.parquet"
        if not done(pf):
            with timer(f"join {split} part {p}"):
                k = pl.scan_parquet(kf).filter(pl.col("k") % PARTS == p).collect()
                sz = k.group_by("k").agg((pl.col("src") == 1).sum().alias("n1"),
                                         (pl.col("src") != 1).sum().alias("n23"))
                good = sz.filter((pl.col("n1") >= 1) & (pl.col("n1") <= CAP1) &
                                 (pl.col("n23") >= 1) & (pl.col("n23") <= CAP23)).select("k")
                k = k.join(good, on="k", how="semi")
                a = k.filter(pl.col("src") == 1).select(pl.col("id").alias("s1"), "k", "kt")
                b = k.filter(pl.col("src") != 1).select("id", "k")
                (a.join(b, on="k").group_by("s1", "id")
                 .agg(*[(pl.col("kt") == t).sum().cast(pl.Int8).alias(f"nk{t}") for t in range(NKT)])
                 .write_parquet(pf))
                del k, a, b
        parts.append(pf)
    # merge per S1 bucket (a pair can come from several key partitions) -> rank -> top-K
    for j in range(BUCKETS):
        if done(cand_path(split, j)):
            continue
        with timer(f"merge {split} bucket {j}"):
            c = (pl.scan_parquet(parts).filter(pl.col("s1") % BUCKETS == j).group_by("s1", "id")
                 .agg(pl.col(f"nk{t}").sum() for t in range(NKT)).collect())
            c = c.with_columns(pl.sum_horizontal([KW[t] * pl.col(f"nk{t}").cast(pl.Int16) for t in range(NKT)])
                               .cast(pl.Int16).alias("sc"))
            c = c.with_columns(pl.col("sc").rank("ordinal", descending=True).over("s1").cast(pl.Int16).alias("rk"),
                               pl.len().over("s1").cast(pl.Int32).alias("ncand_raw"))
            c.filter(pl.col("rk") <= TOPK).write_parquet(cand_path(split, j))
            if split == "train":   # uncapped copy of a single bucket, to measure what top-K loses
                if j == 0:
                    c.select("s1", "id", "sc", "rk").write_parquet(WORK / "train_cand2_full0.parquet")
            del c
    for pf in parts:
        pf.unlink()


def recall_report():
    c = pl.scan_parquet([cand_path("train", j) for j in range(BUCKETS)])
    gt = pl.scan_parquet(WORK / "train_gt.parquet")
    n_s1 = pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1).select(pl.len()).collect().item()
    npairs = c.select(pl.len()).collect().item()
    hit = gt.join(c, on=["s1", "id"], how="inner").select(pl.len()).collect().item()
    ngt = gt.select(pl.len()).collect().item()
    print(f"pairs {npairs:,}  per S1 {npairs / n_s1:.1f}  recall {hit / ngt:.4f} ({hit:,}/{ngt:,})")
    f = pl.read_parquet(WORK / "train_cand2_full0.parquet")
    g0 = gt.filter(pl.col("s1") % BUCKETS == 0).collect()
    h = g0.join(f, on=["s1", "id"])
    print("bucket0 uncapped recall", round(h.height / g0.height, 4),
          {k: round((h["rk"] <= k).sum() / g0.height, 4) for k in (10, 20, 30, 40, 50, 60)})


if __name__ == "__main__":
    for s in (sys.argv[1:] or SPLITS):
        block(s)
    recall_report()

In [ ]:
%%writefile /content/ber/src/e1_emb.py
"""Parallel path E: multilingual-e5-small embeddings for (1) kNN blocking and (2) a cosine feature.
  encode  : "query: <name> | <address>" for every record -> {split}_emb.f16 (memmap [N, 384] float16, L2-normalised)
            row order = {split}_records.parquet row order (ids in {split}_emb_ids.npy)
  knn     : within each country, each S1 -> top-K S2/S3 by cosine (GPU, tiled matmul) -> knn_{split}_{j}.parquet
            [s1, id, cos, erk]   (j = S1 bucket, same bucketing as cand2)
  recall  : train only: recall of cand2, kNN, and their union vs ground truth; new pairs per S1
  cos     : cosine for every existing cand2 pair -> embcos_{split}_{j}.parquet [s1, id, emb_cos, emb_rk]
            (joined automatically by s4_train stage 3 when present)
Every step skips outputs that already exist. Nothing big is held in RAM (memmap + chunks)."""
import os
import sys
import time

import numpy as np
import polars as pl
import torch
from transformers import AutoModel, AutoTokenizer

from common import WORK, done, timer
from s2_block import BUCKETS, cand_path

MODEL = os.environ.get("BER_EMB_MODEL", "/content/drive/MyDrive/MLC'26/models/multilingual-e5-small")
K = int(os.environ.get("BER_KNN_K", 20))
DIM, MAXLEN, BS = 384, 64, 1024
QB = 1024   # S1 rows per GPU block: scores tile = QB x 1M fp16 = 2 GB
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def emb_path(split):
    return WORK / f"{split}_emb.f16"


def ids_path(split):
    return WORK / f"{split}_emb_ids.npy"


def knn_path(split, j):
    return WORK / f"knn_{split}_{j}.parquet"


def embcos_path(split, j):
    return WORK / f"embcos_{split}_{j}.parquet"


def records(split):
    return pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id", "src", "name", "addr", "country"])


def open_emb(split, mode="r"):
    n = len(np.load(ids_path(split)))
    return np.memmap(emb_path(split), dtype=np.float16, mode=mode, shape=(n, DIM))


NCHUNK = 8   # encode in 8 chunks; each finished chunk is saved (and copied to BER_EMB_SYNC if set)


def chunk_path(split, c):
    return WORK / f"{split}_emb_c{c}.f16"


@torch.no_grad()
def encode(split):
    """Resumable: records are encoded shortest-first in NCHUNK chunks. A finished chunk is written as
    {split}_emb_c{c}.f16 and, if BER_EMB_SYNC is set, copied there (Drive) at once, so a crash or a dead session
    loses at most one chunk. Chunks already present are skipped. Finally the chunks are assembled into
    {split}_emb.f16 in records row order (same layout as before)."""
    import shutil
    if done(emb_path(split)) and done(ids_path(split)):
        return
    sync = os.environ.get("BER_EMB_SYNC")
    with timer(f"encode {split}"):
        r = records(split)
        np.save(ids_path(split), r["id"].to_numpy())
        txt = ("query: " + r["name"] + " | " + r["addr"]).str.slice(0, 250).to_list()
        del r
        n = len(txt)
        order = np.argsort(np.fromiter((len(t) for t in txt), dtype=np.int32, count=n), kind="stable")
        bounds = np.linspace(0, n, NCHUNK + 1).astype(np.int64)
        tok = model = None
        for c in range(NCHUNK):
            if done(chunk_path(split, c)):
                print(f"  chunk {c} present, skipped", flush=True)
                continue
            if model is None:
                tok = AutoTokenizer.from_pretrained(MODEL)
                model = AutoModel.from_pretrained(MODEL).to(DEV).half().eval()
            idx_c = order[bounds[c]:bounds[c + 1]]
            tmp = chunk_path(split, c).with_suffix(".part")
            out = np.memmap(tmp, dtype=np.float16, mode="w+", shape=(len(idx_c), DIM))
            t0 = time.time()
            for k in range(0, len(idx_c), BS):
                idx = idx_c[k:k + BS]
                enc = tok([txt[i] for i in idx], padding=True, truncation=True, max_length=MAXLEN,
                          return_tensors="pt").to(DEV)
                h = model(**enc).last_hidden_state
                m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
                e = torch.nn.functional.normalize(((h * m).sum(1) / m.sum(1)).float(), dim=1)
                out[k:k + len(idx)] = e.half().cpu().numpy()
                if k % (BS * 300) == 0:
                    el = time.time() - t0
                    print(f"  chunk {c + 1}/{NCHUNK}: {k:,}/{len(idx_c):,} {(k + BS) / max(el, 1e-6):.0f}/s "
                          f"chunk eta {el / (k + BS) * (len(idx_c) - k) / 60:.0f} min", flush=True)
            out.flush()
            del out
            os.replace(tmp, chunk_path(split, c))
            if sync:
                shutil.copy(chunk_path(split, c), os.path.join(sync, chunk_path(split, c).name + ".part"))
                os.replace(os.path.join(sync, chunk_path(split, c).name + ".part"), os.path.join(sync, chunk_path(split, c).name))
            print(f"  CHUNK {c + 1}/{NCHUNK} DONE ({time.time() - t0:.0f}s){' + saved to Drive' if sync else ''}", flush=True)
        # assemble in records row order (sorted writes -> near-sequential I/O)
        tmp = WORK / f"{split}_emb.f16.part"
        E = np.memmap(tmp, dtype=np.float16, mode="w+", shape=(n, DIM))
        for c in range(NCHUNK):
            pos = order[bounds[c]:bounds[c + 1]]
            C = np.memmap(chunk_path(split, c), dtype=np.float16, mode="r", shape=(len(pos), DIM))
            srt = np.argsort(pos)
            for i in range(0, len(pos), 1_000_000):
                sl = srt[i:i + 1_000_000]
                E[pos[sl]] = C[sl]
            del C
        E.flush()
        del E
        os.replace(tmp, emb_path(split))   # only a finished file gets the final name


@torch.no_grad()
def knn(split):
    if all(done(knn_path(split, j)) for j in range(BUCKETS)):
        return
    E = open_emb(split)
    r = pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id", "src", "country"]).with_row_index("row")
    res = []
    for c in r["country"].unique().to_list():
        with timer(f"knn {split} {c}"):
            q = r.filter((pl.col("country") == c) & (pl.col("src") == 1))
            d = r.filter((pl.col("country") == c) & (pl.col("src") != 1))
            if q.height == 0 or d.height == 0:
                continue
            drow, did = d["row"].to_numpy(), d["id"].to_numpy()
            # database on GPU in chunks of 1M rows (fp16: ~0.77 GB per chunk)
            D = [torch.from_numpy(np.ascontiguousarray(E[np.sort(drow[i:i + 1_000_000])])).to(DEV)
                 for i in range(0, len(drow), 1_000_000)]
            dorder = np.concatenate([np.sort(drow[i:i + 1_000_000]) for i in range(0, len(drow), 1_000_000)])
            row2id = dict(zip(drow.tolist(), did.tolist()))
            did_sorted = np.fromiter((row2id[x] for x in dorder), dtype=np.int64, count=len(dorder))
            qrow, qid = q["row"].to_numpy(), q["id"].to_numpy()
            for i in range(0, len(qrow), QB):
                Q = torch.from_numpy(np.ascontiguousarray(E[qrow[i:i + QB]])).to(DEV)
                best_s, best_i, off = [], [], 0
                for Dc in D:
                    s = Q @ Dc.T
                    v, ix = s.topk(min(K, Dc.shape[0]), dim=1)
                    best_s.append(v.float()); best_i.append(ix + off); off += Dc.shape[0]
                v, ix = torch.cat(best_s, 1).topk(K, dim=1)
                gi = torch.gather(torch.cat(best_i, 1), 1, ix).cpu().numpy()
                v = v.cpu().numpy()
                res.append(pl.DataFrame({"s1": np.repeat(qid[i:i + QB], K), "id": did_sorted[gi.ravel()],
                                         "cos": v.ravel().astype(np.float32),
                                         "erk": np.tile(np.arange(1, K + 1, dtype=np.int16), len(gi))}))
            del D
            torch.cuda.empty_cache()
    res = pl.concat(res)
    for j in range(BUCKETS):
        res.filter(pl.col("s1") % BUCKETS == j).write_parquet(knn_path(split, j))


def recall():
    """Bucket by bucket (loading all 110M candidate pairs at once crashed a 12.7 GB runtime)."""
    tot = hc = n1 = ncand = 0
    hk, hu, nu = {5: 0, 10: 0, 20: 0}, {5: 0, 10: 0, 20: 0}, {5: 0, 10: 0, 20: 0}
    for j in range(BUCKETS):
        gt = pl.read_parquet(WORK / "train_gt.parquet").filter(pl.col("s1") % BUCKETS == j)
        cand = pl.read_parquet(cand_path("train", j), columns=["s1", "id"])
        kn = pl.read_parquet(knn_path("train", j))
        n1 += cand["s1"].n_unique(); ncand += cand.height; tot += gt.height
        hc += gt.join(cand, on=["s1", "id"], how="semi").height
        for k in (5, 10, 20):
            kk = kn.filter(pl.col("erk") <= k).select("s1", "id")
            u = pl.concat([cand, kk]).unique()
            hk[k] += gt.join(kk, on=["s1", "id"], how="semi").height
            hu[k] += gt.join(u, on=["s1", "id"], how="semi").height
            nu[k] += u.height - cand.height
        del cand, kn
    print(f"cand2 recall {hc / tot:.4f}")
    for k in (5, 10, 20):
        print(f"kNN@{k}: alone {hk[k] / tot:.4f} | union {hu[k] / tot:.4f} | new pairs/S1 {nu[k] / n1:.1f}", flush=True)


def cosine(split):
    E = open_emb(split)
    row = pl.DataFrame({"id": np.load(ids_path(split))}).with_row_index("row")
    for j in range(BUCKETS):
        if done(embcos_path(split, j)):
            continue
        with timer(f"cos {split} {j}"):
            c = (pl.read_parquet(cand_path(split, j), columns=["s1", "id"])
                 .join(row.rename({"id": "s1", "row": "r1"}), on="s1").join(row.rename({"row": "r2"}), on="id"))
            out = np.empty(c.height, dtype=np.float32)
            r1, r2 = c["r1"].to_numpy(), c["r2"].to_numpy()
            for i in range(0, c.height, 1_000_000):
                a = torch.from_numpy(E[r1[i:i + 1_000_000]]).to(DEV).float()
                b = torch.from_numpy(E[r2[i:i + 1_000_000]]).to(DEV).float()
                out[i:i + 1_000_000] = (a * b).sum(1).cpu().numpy()
            (c.select("s1", "id").with_columns(pl.Series("emb_cos", out))
             .with_columns(pl.col("emb_cos").rank("min", descending=True).over("s1").cast(pl.Int16).alias("emb_rk"))
             .write_parquet(embcos_path(split, j)))


if __name__ == "__main__":
    # usage: e1_emb.py encode|knn|recall|cos <split...>
    step, splits = sys.argv[1], sys.argv[2:] or ["train", "test"]
    for s in splits:
        {"encode": encode, "knn": knn, "cos": cosine}.get(step, lambda _: None)(s)
    if step == "recall":
        recall()

In [ ]:
run_emb = r"""#!/bin/bash
set -e
D="/content/drive/MyDrive/MLC'26"
cd /content/ber/src
OUT="$D/work_emb"
export BER_WORK=/content/work BER_EMB_MODEL="$D/models/multilingual-e5-small" BER_EMB_SYNC="$OUT"
echo "== encode test $(date)"
python e1_emb.py encode test
[ "$(stat -c%s "$OUT/test_emb.f16" 2>/dev/null)" = "$(stat -c%s /content/work/test_emb.f16)" ] || cp /content/work/test_emb.f16 /content/work/test_emb_ids.npy "$OUT/"
[ "$(stat -c%s "$OUT/test_emb.f16")" = "$(stat -c%s /content/work/test_emb.f16)" ] && rm -f "$OUT"/test_emb_c*.f16   # full file safe on Drive -> drop the chunks
echo "== knn test $(date)"
python e1_emb.py knn test
cp /content/work/knn_test_*.parquet "$OUT/"
echo "== EMB TEST DONE $(date)"
"""
open("/content/run_emb.sh", "w").write(run_emb)
sh('bash -n /content/run_emb.sh && echo syntax-ok')

## Cell 4: launch in the background

In [ ]:
subprocess.Popen("setsid nohup bash /content/run_emb.sh > /content/log_emb.txt 2>&1 < /dev/null &", shell=True)  # setsid: survives kernel restarts
import time; time.sleep(10); sh("pgrep -af 'run_emb' | grep -v pgrep; tail -2 /content/log_emb.txt")

## Cell 5: status (re-run any time)

In [ ]:
sh(f"""echo "=== $(date -u +%H:%M) UTC"
if pgrep -f 'run_[e]mb' >/dev/null; then echo "JOB: RUNNING"; else echo "JOB: NOT RUNNING"; fi
grep -E '^> |: [0-9]+s$|==' /content/log_emb.txt | tail -4; tail -1 /content/log_emb.txt
echo "--- errors?"; grep -iE 'error|killed|traceback|out of memory' /content/log_emb.txt | tail -3 || true
echo "--- on Drive (work_emb): chunks $(ls "{D}/work_emb" | grep -c 'test_emb_c[0-7].f16$')/8, test_emb.f16 $(stat -c%s "{D}/work_emb/test_emb.f16" 2>/dev/null || echo missing) bytes, knn_test $(ls "{D}/work_emb" | grep -c knn_test)/8"
free -m | awk 'NR==2{{print "  RAM used " $3 " MB, available " $7 " MB"}}'
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader | awk '{{print "  GPU " $0}}'""")

## Done when
Cell 5 shows `== EMB TEST DONE`, the Drive `test_emb.f16` at **~9.0 GB** (matching the local file), and `knn_test 8/8`. Then delete the runtime.

## If something goes wrong
- `JOB: NOT RUNNING` without `EMB TEST DONE`: re-run cell 4 (finished steps are skipped). Send the last 30 lines of `/content/log_emb.txt` if it fails again.
- GPU out of memory in kNN: in cell 3 change `QB = 1024` to `QB = 512` in e1_emb.py, re-run cell 3, then cell 4.

In [ ]:
# ---- clean stop ----
sh("for p in $(pgrep -f 'run_[e]mb.sh'); do kill $p; done; sleep 1; for p in $(pgrep -f 'python [e]1_emb'); do kill $p; done; sleep 2; pgrep -af python | grep -v colab")